In [ ]:
!pip install -q "shekar" "ipywidgets" "transformers>=4.41" "datasets>=2.19" "accelerate>=0.30" sacrebleu jiwer sentencepiece

In [ ]:
import os
import random
import numpy as np
import torch
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)
import sacrebleu
from jiwer import cer

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
MODEL_NAME = "google/byt5-small"
DATA_DIR = "./data"  # Path to merged ParsText dataset tsv file
OUTPUT_DIR = "./byt5-small-tajik-farsi"

In [ ]:
import pandas as pd
from datasets import Dataset

ALL_DATA_FILE = os.path.join(DATA_DIR, "all_data.tsv")

df = pd.read_csv(
    ALL_DATA_FILE, sep="\t", header=None, names=["tajik", "persian"],
    on_bad_lines="skip", dtype=str,
)
print(f"Rows loaded: {len(df)}")

ds = Dataset.from_pandas(df, preserve_index=False)
print("Dataset:", ds)
print("\nFirst example:")
print(ds[0])

In [ ]:
keep_cols = ["tajik", "persian"]
drop_cols = [c for c in ds.column_names if c not in keep_cols]
ds = ds.remove_columns(drop_cols)

def is_valid(ex):
    t = ex.get("tajik") or ""
    p = ex.get("persian") or ""
    t, p = t.strip(), p.strip()
    # Skip empty rows and absurdly long ones that would just waste compute
    return len(t) > 0 and len(p) > 0 and len(t) <= 200 and len(p) <= 200

before = len(ds)
ds = ds.filter(is_valid)
print(f"Filtered: {before} -> {len(ds)} rows")

In [ ]:
from shekar.preprocessing import AlphabetNormalizer, YaNormalizer, SpacingNormalizer

normalization_pipeline = AlphabetNormalizer() | YaNormalizer(style="standard") | SpacingNormalizer()
test = "آبشوي"
print(normalization_pipeline(test))
test = "سالها"
print(normalization_pipeline(test)) 
test = "سال ها"
print(normalization_pipeline(test))


In [ ]:
def expand_bidirectional(batch):
    srcs, tgts, dirs = [], [], []
    for fa, tg in zip(batch["persian"], batch["tajik"]):
        fa, tg = fa.strip(), tg.strip()

        # normalize Persian text
        fa = normalization_pipeline(fa)

        # fa -> tg
        srcs.append(f"fa2tg: {fa}")
        tgts.append(tg)
        dirs.append("fa2tg")
        # tg -> fa
        srcs.append(f"tg2fa: {tg}")
        tgts.append(fa)
        dirs.append("tg2fa")
    return {"source": srcs, "target": tgts, "direction": dirs}

ds_bi = ds.map(
    expand_bidirectional,
    batched=True,
    remove_columns=ds.column_names,
    desc="Expanding to bidirectional examples",
)
print(ds_bi)
print("\nSample:")
for i in [0, 1, 2, 3]:
    print(ds_bi[i])

In [ ]:
ds_bi = ds_bi.shuffle(seed=SEED)
n = len(ds_bi)
n_train = int(n * 0.90)
n_val = int(n * 0.05)
n_test = int(n * 0.05)

splits = DatasetDict({
    "train": ds_bi.select(range(0, n_train)),
    "validation": ds_bi.select(range(n_train, n_train + n_val)),
    "test": ds_bi.select(range(n_train + n_val, n_train + n_val + n_test)),
})
print(splits)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer:", tokenizer.__class__.__name__)
print("Vocab size:", tokenizer.vocab_size)

sample_fa = "fa2tg: کتاب"
sample_tg = "tg2fa: китоб"
print(f"\n{sample_fa!r} -> {len(tokenizer(sample_fa).input_ids)} tokens")
print(f"{sample_tg!r} -> {len(tokenizer(sample_tg).input_ids)} tokens")

In [ ]:
def preprocess(batch):
    model_inputs = tokenizer(
        batch["source"],
        max_length=1024,
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch["target"],
        max_length=1024,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = splits.map(
    preprocess,
    batched=True,
    remove_columns=["source", "target"],   # keep 'direction' for per-direction eval
    desc="Tokenizing",
)
print(tokenized)

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model: {MODEL_NAME}")
print(f"Parameters: {n_params/1e6:.1f}M")

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding="longest",
    label_pad_token_id=-100,
)

In [ ]:
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Replace -100 in labels (used to mask padding) with the pad token before decoding
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    chrf_pp = sacrebleu.corpus_chrf(
        decoded_preds, [decoded_labels], word_order=2
    ).score  # chrF++ = chrF with word_order=2

    # jiwer.cer wants non-empty refs; guard just in case
    refs = [l if len(l) > 0 else " " for l in decoded_labels]
    hyps = [p if len(p) > 0 else " " for p in decoded_preds]
    cer_score = cer(refs, hyps)

    # Exact match (sequence accuracy)
    exact = np.mean([p == l for p, l in zip(decoded_preds, decoded_labels)])

    return {
        "chrF++": round(chrf_pp, 2),
        "CER": round(cer_score, 4),
        "seq_acc": round(float(exact), 4),
    }

In [ ]:
use_fp16 = torch.cuda.is_available()

use_bf16 = use_fp16 and torch.cuda.is_bf16_supported()
if use_bf16:
    use_fp16 = False

print(f"Using mixed precision: fp16={use_fp16}, bf16={use_bf16}")

args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    learning_rate=0.0005,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type="linear",
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=5000,
    save_strategy="steps",
    save_steps=5000,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="chrF++",
    greater_is_better=True,
    predict_with_generate=True,
    generation_max_length=1024,
    generation_num_beams=4,
    fp16=use_fp16,
    bf16=use_bf16,
    report_to="none",
    seed=SEED,
)

train_cols_to_drop = [c for c in tokenized["train"].column_names if c == "direction"]
tokenized_for_trainer = tokenized.remove_columns(train_cols_to_drop)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_for_trainer["train"],
    eval_dataset=tokenized_for_trainer["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(train_result.metrics)

In [ ]:
# Overall test metrics
test_metrics = trainer.evaluate(
    eval_dataset=tokenized_for_trainer["test"],
    metric_key_prefix="test",
)
print("\n=== Overall test ===")
for k, v in test_metrics.items():
    print(f"  {k}: {v}")

In [ ]:
# Per-direction breakdown (fa2tg vs tg2fa)
for direction in ["fa2tg", "tg2fa"]:
    subset_indices = [i for i, d in enumerate(tokenized["test"]["direction"]) if d == direction]
    subset = tokenized_for_trainer["test"].select(subset_indices)
    metrics = trainer.evaluate(eval_dataset=subset, metric_key_prefix=f"test_{direction}")
    print(f"\n=== {direction} (n={len(subset)}) ===")
    for k, v in metrics.items():
        if any(m in k for m in ["chrF++", "CER", "seq_acc", "loss"]):
            print(f"  {k}: {v}")

In [ ]:
#login to Hugging Face Hub (optional, for pushing the model later)

# from huggingface_hub import login
# login()

In [ ]:
from pathlib import Path

BEST_CHECKPOINT = Path(OUTPUT_DIR) / "best"

model = AutoModelForSeq2SeqLM.from_pretrained(BEST_CHECKPOINT)
tokenizer = AutoTokenizer.from_pretrained(BEST_CHECKPOINT)

# #push to hub 
# model.push_to_hub("shekar-ai/byt5-small-tajik-farsi-translit")
# tokenizer.push_to_hub("shekar-ai/byt5-small-tajik-farsi-translit")

Quick inference helper. Pass any Persian or Tajik string and a direction.

In [ ]:
device = next(model.parameters()).device
model.eval()

@torch.no_grad()
def transliterate(text, direction="fa2tg", num_beams=4, max_new_tokens=1024):
    assert direction in {"fa2tg", "tg2fa"}
    src = f"{direction}: {text.strip()}"
    inputs = tokenizer(src, return_tensors="pt", truncation=True, max_length=1024).to(device)
    out = model.generate(
        **inputs,
        num_beams=num_beams,
        max_new_tokens=max_new_tokens,
        early_stopping=True,
    )
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()

examples = [
    ("کتاب", "fa2tg"),
    ("دانشگاه", "fa2tg"),
    ("китоб", "tg2fa"),
    ("донишгоҳ", "tg2fa"),
]
for txt, d in examples:
    print(f"[{d}] {txt!r:>20}  ->  {transliterate(txt, d)!r}")

In [ ]:
# %pip install -q "optimum[onnxruntime]>=1.20" "onnx>=1.16" "onnxruntime>=1.18"

In [ ]:
import gc, time, shutil
from pathlib import Path
from optimum.onnxruntime import ORTModelForSeq2SeqLM

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

MERGED_ROOT = Path(OUTPUT_DIR + "-merged-onnx")
MERGED_FP32 = MERGED_ROOT / "fp32"
MERGED_INT8 = MERGED_ROOT / "int8"
for p in (MERGED_FP32, MERGED_INT8):
    p.mkdir(parents=True, exist_ok=True)

t0 = time.time()
ort_model_merged = ORTModelForSeq2SeqLM.from_pretrained(
    str(BEST_CHECKPOINT),
    export=True,
    use_cache=True,
    use_merged=True,
)
ort_model_merged.save_pretrained(MERGED_FP32)
tokenizer.save_pretrained(MERGED_FP32)
print(f"FP32 merged ONNX export done in {time.time() - t0:.1f}s -> {MERGED_FP32}")
print("Files:", sorted(p.name for p in MERGED_FP32.iterdir()))


In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

for f in MERGED_FP32.iterdir():
    if f.suffix != ".onnx" and f.is_file():
        shutil.copy2(f, MERGED_INT8 / f.name)

onnx_files = sorted(MERGED_FP32.glob("*.onnx"))
print(f"Quantizing {len(onnx_files)} ONNX graphs to INT8 (dynamic, per-channel)...")
for src in onnx_files:
    dst = MERGED_INT8 / src.name
    is_encoder = "encoder" in src.name
    quantize_dynamic(
        model_input=str(src),
        model_output=str(dst),
        weight_type=QuantType.QInt8,
        per_channel=True,
        reduce_range=is_encoder,   # True for encoder (safer on AVX2), False for decoder
    )
    src_mb = src.stat().st_size / 1e6
    dst_mb = dst.stat().st_size / 1e6
    tag = "reduce_range=True" if is_encoder else "reduce_range=False"
    print(f"  {src.name} ({tag}): {src_mb:.1f} MB -> {dst_mb:.1f} MB")

print(f"\nINT8 export at: {MERGED_INT8}")
print("Files:", sorted(p.name for p in MERGED_INT8.iterdir()))